# Final CHO Model
This notebook is to asses the validity of our reconstruction and how complete it is.

[1. Generation of the dataset and model reconstruction](#generation) <br>
&nbsp;&nbsp;&nbsp;&nbsp;**1.1 Retrieve information from the Google Sheet datasets reactions and metabolites**<br>
&nbsp;&nbsp;&nbsp;&nbsp;**1.2 Build a model and feed it the information from the df generated** <br>
&nbsp;&nbsp;&nbsp;&nbsp;**1.3 Save and validate the model** <br>
&nbsp;&nbsp;&nbsp;&nbsp;**1.4 Check for unbalanced reactions** <br>

[2. Identification of Blocked Reactions and Dead-End Metabolites](#blocked&deadends) <br>
&nbsp;&nbsp;&nbsp;&nbsp;**2.1 Identification of Blocked Reactions**<br>
&nbsp;&nbsp;&nbsp;&nbsp;**2.2 Identification of Dead-Ends Metabolites** <br>
&nbsp;&nbsp;&nbsp;&nbsp;**2.3 Gap-fill** <br>
&nbsp;&nbsp;&nbsp;&nbsp;**2.4 Addition of Extracellular Exchange Reanctions** <br>

## 1. Generation of the dataset and model reconstruction <a id='generation'></a>
Here we generate the CHO model from the dataset stored in the Google Sheet file. We first use the google_sheet module to extract all the necessary information from the original dataset. Then we use those dataset and the COBRApy library to: (1) Create a new model and add reactions from the **Rxns Sheet**, (2) Add information on each reaction obtained from the **Rxns Sheet** and **Attributes Sheet**, (3) Add boundary reactions from the **BoundaryRxns Sheet**, and (4) Add information for each metabolite from the **Metabolites Sheet**. Finally we save the model as a SBML file and validate it using the cobrapy built-in function "validate_sbml_model( )".

In [1]:
# Import libraries
import pandas as pd
import numpy as np

import cobra
from cobra import Model, Reaction, Metabolite
from cobra.io import validate_sbml_model, save_json_model, write_sbml_model, save_matlab_model, load_matlab_model

from tqdm.notebook import tqdm

### 1.1 Retrieve information from iCHO3K datasets

In [2]:
##### ----- Read iCHO3K reaction files ----- #####

#Path to iCHO3K Excell
FILE_PATH = '../iCHO3K/Dataset/iCHO3K.xlsx'


# Sheets
sheet_met = 'Metabolites'
sheet_rxns = 'Rxns'
sheet_attributes = 'Attributes'
sheet_boundary = 'BoundaryRxns'
sheet_genes = 'Genes'

# Read into DataFrames
metabolites      = pd.read_excel(FILE_PATH, sheet_name=sheet_met)
rxns             = pd.read_excel(FILE_PATH, sheet_name=sheet_rxns)
rxns_attributes  = pd.read_excel(FILE_PATH, sheet_name=sheet_attributes)
boundary_rxns    = pd.read_excel(FILE_PATH, sheet_name=sheet_boundary)
genes_df         = pd.read_excel(FILE_PATH, sheet_name=sheet_genes)

### 1.2 Build a model and feed it the information from the df generated

In [3]:
##### ----- Create a model and add reactions ----- #####
model = Model("iCHO3K")
lr = []
for _, row in rxns.iterrows():
    r = Reaction(row['Reaction'])
    lr.append(r)    
model.add_reactions(lr)

Set parameter Username
Academic license - for non-commercial use only - expires 2027-03-16


In [ ]:
##### ----- Add information to each one of the reactions ----- #####

for i,r in enumerate(tqdm(model.reactions)):
    r.build_reaction_from_string(rxns['Reaction Formula'][i])
    r.name = rxns['Reaction Name'][i]
    r.subsystem = rxns['Subsystem'][i]
    if not (pd.isna(rxns['GPR_iCHO3K'][i]) or rxns['GPR_iCHO3K'][i] == ''):
        r.gene_reaction_rule = str(rxns['GPR_iCHO3K'][i])
    r.lower_bound = float(rxns_attributes['Lower bound'][i])
    r.upper_bound = float(rxns_attributes['Upper bound'][i])
    r.annotation['confidence_score'] = str(rxns['Conf. Score'][i])
    r.annotation['molwt'] = str(rxns_attributes['Mol wt'][i])
    r.annotation['kcat_f'] = str(rxns_attributes['kcat_forward'][i])
    r.annotation['kcat_b'] = str(rxns_attributes['kcat_backward'][i])

In [5]:
##### ----- Add Boundary Reactions ----- #####
dr = []
for _, row in boundary_rxns.iterrows():
    r = Reaction(row['Reaction'])
    dr.append(r)    
model.add_reactions(dr)

boundary_rxns_dict = boundary_rxns.set_index('Reaction').to_dict()
boundary_rxns_dict

for i,r in enumerate(tqdm(model.reactions)):
    if r in dr:
        r.build_reaction_from_string(boundary_rxns_dict['Reaction Formula'][r.id])
        r.name = boundary_rxns_dict['Reaction Name'][r.id]
        r.subsystem = boundary_rxns_dict['Subsystem'][r.id]
        r.lower_bound = float(boundary_rxns_dict['Lower bound'][r.id])
        r.upper_bound = float(boundary_rxns_dict['Upper bound'][r.id])
        r.annotation['confidence_score'] = str(1)

  0%|          | 0/11004 [00:00<?, ?it/s]

In [6]:
##### ----- Add information for each metabolite ----- #####
metabolites_dict = metabolites.set_index('BiGG ID').to_dict('dict')
for met in model.metabolites:
    try:
        met.name = metabolites_dict['Name'][f'{met}']
        met.formula = metabolites_dict['Formula'][f'{met}']
        met.compartment = metabolites_dict['Compartment'][f'{met}'].split(' - ')[0]
        try:
            met.charge = int(metabolites_dict['Charge'][f'{met}'])
        except (ValueError, TypeError):
            print(f'{met} doesnt have charge')
    except (KeyError):
        print('----------------------------')
        print(f'{met} doesnt exist in the df')
        print('----------------------------')

dnac_n doesnt have charge
lipACP_c doesnt have charge


In [7]:
##### ----- Add Gene Name information ----- #####
genes_dict = genes_df.iloc[:,:2].set_index('Gene Entrez ID').to_dict('dict')
for g in model.genes:
    if g.id in list(genes_dict['Gene Symbol'].keys()):
        g.name = genes_dict['Gene Symbol'][f'{g}']

### 1.3 Save and validate the model

In [8]:
name  = 'CHO-Generic'  # or 'CHO-S', or 'CHO-Generic'

In [9]:
##### ----- Build the S matrix ----- #####

model.solver = 'gurobi'

if name == 'CHO-Generic':
    model.objective = 'biomass_cho'
elif name == 'CHO-S':
    model.objective = 'biomass_cho_s'
elif name == 'CHO-Prod-Generic':
    model.objective = 'biomass_cho_prod'
else:
    raise ValueError(f"Unrecognized model name: {name}")
    
    
S = cobra.util.create_stoichiometric_matrix(model, array_type='dense')
model.S = S
    
sol = model.optimize()
sol

,fluxes,reduced_costs
AGTim,0.0,0.0
AGTix,0.0,0.0
ARGSL,0.0,0.0
ARGSS,0.0,0.0
ASNN,0.0,0.0
...,...,...
EX_CN0020_e,0.0,0.0
EX_M00932_e,0.0,0.0
EX_M00545_e,0.0,0.0
EX_M00228_e,0.0,0.0


In [10]:
##### ----- Save the entiry reconstruction witouh any preprocessing ----- #####

suffix = name.lower().replace('-', '_')

# XML
model_name_xml = f'../iCHO3K/Model/iCHO3K_{suffix}.xml' 
write_sbml_model(model, model_name_xml)

# JSON, because the sbml doesnt save the subsystems
model_name_json = f'../iCHO3K/Model/iCHO3K_{suffix}.json' 
save_json_model(model, model_name_json)

# MATLAB
model_name_matlab = f'../iCHO3K/Model/iCHO3K_{suffix}.mat' 
save_matlab_model(model, model_name_matlab)

In [11]:
##### ----- Test for errors in the recostruction ----- ######

# import tempfile
# from pprint import pprint
# from cobra.io import write_sbml_model, validate_sbml_model
# with tempfile.NamedTemporaryFile(suffix='.xml') as f_sbml:
#     write_sbml_model(model, filename=f_sbml.name)
#     report = validate_sbml_model(filename=f_sbml.name)
# pprint(report)

from cobra.io import read_sbml_model, validate_sbml_model
(_, errors) = validate_sbml_model(model_name_xml)
errors

{'SBML_FATAL': [],
 'SBML_ERROR': [],
 'SBML_SCHEMA_ERROR': [],
 'SBML_WARNING': [],
 'COBRA_FATAL': [],
 'COBRA_ERROR': [],
 'COBRA_WARNING': [],
 'COBRA_CHECK': []}

## 2. Identification of Blocked Reactions and Dead-End Metabolites <a id='blocked&deadends'></a>
In this second part of the notebook we use two different functions from the utils module to: (1) Run a flux variability analysis and identify blocked reactions, and (2) identify dead-end metabolites. Finally we add Extracellular Exchange reactions for the dead-end metabolites that are in the extracellular compartment.

In [12]:
import pandas as pd
from cobra.io import load_json_model, read_sbml_model, load_matlab_model
from cobra.flux_analysis import find_blocked_reactions, flux_variability_analysis
from Utils.utils import detect_dead_ends

In [13]:
##### ----- Read Model ----- #####
if 'model' not in locals():
    model = load_json_model(f"iCHO3K_{suffix}.json")
    print('Model loaded')
else:
    print('Model already generated')

Model already generated


### 2.1 Identification of Blocked Reactions
Here we use the COBRApy built-in functions **find_blocked_reactions** and **flux_variability_analysis** to find blocked reactions in our reconstruction and run an FVA analysis respectively.

In [18]:
# Run this code to remove glucose generating loop

glucloop_reactions = [
    'GapFill-R01206', 'GAUGE-R00557', 'GLYC3PFADm', 
    'GAUGE-R00558', 'FNOR', 'GGH', 'r0741', 'r1479', 'XOLESTPOOL',
]

try:
    model.remove_reactions(glucloop_reactions, remove_orphans=True)
except KeyError:
    print(f'Reaction {reaction_id} not in model {model.id}')

In [19]:
# Run this code to remove reactions that alter the normal TCA cycle

tca_affecting_reactions = [
    'r0082', 'r0083', 'r0084'
]

try:
    model.remove_reactions(tca_affecting_reactions, remove_orphans=True)
except KeyError:
    print(f'Reaction {reaction_id} not in model {model.id}')

In [20]:
# Run this code to remove ATP generating loop

atp_loop_reactions = [
    'SCP22x','TMNDNCCOAtx','OCCOAtx','r0391','BiGGRxn67','r2247','r2280',
    'r2246','r2279','r2245','r2305','r2317','r2335','HMR_0293','HMR_7741',
    'r0509','r1453','HMR_4343','ACONTm','PDHm','r0426','r0383','r0555',
    'r1393','NICRNS','GAUGE-R00648','GAUGE-R03326','GapFill-R08726','RE2915M',
    'HMR_3288','HMR_1325','HMR_7599','r1431','r1433','RE2439C','r0791',
    'r1450','GAUGE-R00270','GAUGE-R02285','GAUGE-R04283','GAUGE-R06127','GAUGE-R06128',
    'GAUGE-R06238','GAUGE-R00524','RE3477C','AAPSAS','RE3347C','HMR_0960','HMR_0980',
    'RE3476C','r0708','r0777','r0424','r0698','3HDH260p','HMR_3272','ACOAD183n3m',
    'HMR_1996','GapFill-R01463','GapFill-R04807','r1468','r2435','r0655','r0603','r0541',
    'RE0383C','HMR_1329','TYRA','NRPPHRt_2H','GAUGE-R07364','GapFill-R03599','ARD',
    'RE3095C','RE3104C','RE3104R','ACONT','ICDHxm','ICDHy',
    'r0425','r0556','NH4t4r','PROPAT4te','r0085','r0156','r0464','ABUTDm',
    'OIVD1m','OIVD2m','OIVD3m','r2194','r2202','HMR_9617','r2197','r2195',
    '2OXOADOXm','r2328','r0386','r0451','FAS100COA','FAS120COA','FAS140COA',
    'FAS80COA_L','r0604','r0670','r2334','r0193','r0595','r0795','GLYCLm',
    'MACACI','r2193','r0779','r0669','UDCHOLt','r2146','r2139'
]

try:
    model.remove_reactions(atp_loop_reactions, remove_orphans=True)
except KeyError:
    print(f'Reaction {reaction_id} not in model {model.id}')

In [21]:
##### ----- Blocked Reactions ----- #####
for rxn in model.boundary:
    if rxn.id.startswith("EX_"):
        rxn.bounds = (-1000,1000)
    if rxn.id.startswith("SK_"):
        rxn.bounds = (-1000,1000)
    if rxn.id.startswith("DM_"):
        rxn.bounds = (0,1000)

model.solver = 'gurobi'
blocked_reactions = find_blocked_reactions(model)

Set parameter Username
Academic license - for non-commercial use only - expires 2027-03-16
Read LP format model from file /var/folders/_x/tfg8s2ks4n1ftkkwzp5sqjpc0000gn/T/tmpuqcw6t5o.lp
Reading time = 0.03 seconds
: 7365 rows, 21753 columns, 91733 nonzeros
Set parameter Username
Academic license - for non-commercial use only - expires 2027-03-16
Read LP format model from file /var/folders/_x/tfg8s2ks4n1ftkkwzp5sqjpc0000gn/T/tmpdyxsrqyx.lp
Reading time = 0.04 seconds
: 7365 rows, 21753 columns, 91733 nonzeros
Set parameter Username
Academic license - for non-commercial use only - expires 2027-03-16
Read LP format model from file /var/folders/_x/tfg8s2ks4n1ftkkwzp5sqjpc0000gn/T/tmp6v7f87_x.lp
Reading time = 0.04 seconds
: 7365 rows, 21753 columns, 91733 nonzeros
Set parameter Username
Academic license - for non-commercial use only - expires 2027-03-16
Read LP format model from file /var/folders/_x/tfg8s2ks4n1ftkkwzp5sqjpc0000gn/T/tmpgokscyly.lp
Reading time = 0.03 seconds
: 7365 rows, 21

In [22]:
### Set the bounds for context-specific model generation

for rxn in model.boundary:
        
    # Models that are forced to secrete ethanol are not feasible
    if rxn.id == 'EX_etoh_e':
        rxn.bounds = (-1,1)
        continue
    
    # Keep boundaries open for essential metabolites
    if rxn.id == 'EX_h2o_e':
        rxn.bounds = (-1000,1000)
        continue
    if rxn.id == 'EX_h_e':
        rxn.bounds = (-1000,1000)
        continue
    if rxn.id == 'EX_o2_e':
        rxn.bounds = (-1000,1000)
        continue
    if rxn.id == 'EX_hco3_e':
        rxn.bounds = (-1000,1000)
        continue
    if rxn.id == 'EX_so4_e':
        rxn.bounds = (-1000,1000)
        continue
    if rxn.id == 'EX_pi_e':
        rxn.bounds = (-1000,1000)
        continue

    # Boundaries from Sink reactions on iCHO_v1 (100 times lower)
    if rxn.id == 'SK_Asn_X_Ser_Thr_r':
        rxn.bounds = (-0.001,1000)
        continue
    if rxn.id == 'SK_Tyr_ggn_c':
        rxn.bounds = (-0.001,1000)
        continue
    if rxn.id == 'SK_Ser_Thr_g':
        rxn.bounds = (-0.001,1000)
        continue
    if rxn.id == 'SK_pre_prot_r':
        rxn.bounds = (-0.001,1000)
        continue
    
    # Close uptake rates for the rest of the boundaries
    if rxn.id.startswith("EX_"):
        rxn.bounds = (0,1000) 
    if rxn.id.startswith("SK_"):
        rxn.bounds = (0,1000)
    if rxn.id.startswith("DM_"):
        rxn.bounds = (0,1000)

In [23]:
### ---- Remove blocked reactions from the model and save it as a separete model ---- ####

model_unblocked = model.copy()

# Convert list of reaction IDs to reaction objects
blocked_reaction_objects = [model_unblocked.reactions.get_by_id(rxn_id) for rxn_id in blocked_reactions]

# Remove blocked reactions
model_unblocked.remove_reactions(blocked_reaction_objects, remove_orphans=True)
print(f"Removed {len(blocked_reaction_objects)} blocked reactions from the model.")

# XML
model_name_xml = f'../iCHO3K/Model/iCHO3K_{suffix}_unblocked.xml' 
write_sbml_model(model_unblocked, model_name_xml)

# JSON, because the sbml doesnt save the subsystems
model_name_json = f'../iCHO3K/Model/iCHO3K_{suffix}_unblocked.json' 
save_json_model(model_unblocked, model_name_json)

# MATLAB
model_name_matlab = f'../iCHO3K/Model/iCHO3K_{suffix}_unblocked.mat' 
save_matlab_model(model_unblocked, model_name_matlab)

Read LP format model from file /var/folders/_x/tfg8s2ks4n1ftkkwzp5sqjpc0000gn/T/tmpexeyg3th.lp
Reading time = 0.04 seconds
: 7364 rows, 21752 columns, 91730 nonzeros
Removed 2508 blocked reactions from the model.


In [24]:
# Save confidence score for Context-Specific model generation

conf_scores = []
for r in model_unblocked.reactions:
    conf_scores.append(r.annotation['confidence_score'])
conf_scores_array = np.array(conf_scores)

np.savetxt(f"../Data/Context_specific_models/confidence_scores{suffix}.csv", conf_scores_array, delimiter=",", fmt='%s')

In [ ]:
### ---- FVA ---- ####
model.solver = 'gurobi'
fva_results = flux_variability_analysis(model)
fva_results.to_excel(f'temp/fva_results{suffix}.xlsx')

In [ ]:
## Check if any reaction in the BIOMASS subsystem is blocked
for rxn in blocked_reactions:
    r = model.reactions.get_by_id(rxn)
    if r.subsystem == 'BIOMASS':
        print(r.id)
        print('-----------------')
        print('-----------------')
        for met in r.metabolites:
            m = model.metabolites.get_by_id(met.id)
            print(m)
            print('.................')
            for r2 in m.reactions:
                if r2.id in blocked_reactions:
                    print(f'No Flux -> {r2.id}: {r2.reaction}')
                else:
                    print(f'With Flux -> {r2.id}: {r2.reaction}')
            print('.................')
            print(' ')
                

In [ ]:
##### ----- Print the amount  and % of blocked reactions ----- #####
print('##### ----- Blocked Reactions ----- #####')
print(f'The model has {len(model.reactions)} total reactions')
print(f'The model has {len(blocked_reactions)} ({round(len(blocked_reactions)/len(model.reactions)*100)}%) blocked reactions')

### 2.2 Identification of Dead-Ends Metabolites
The detect_dead_ends( ) function from the utils module returns a list with all the **dead-end** metabolites in our model. A dead-end metabolite refers to a metabolite that is either only consumed but not produced, or only produced but not consumed, in a given metabolic network. The results are stored in the "Dead-ends.txt" file.

In [ ]:
##### ----- Detect Dead-Ends ----- #####
model.solver = 'gurobi' #change 'gurobi' for the default cobrapy solver 'glpk' 
dead_ends = detect_dead_ends(model)

In [ ]:
counter=0
for i, is_dead_end in enumerate(dead_ends):
    if is_dead_end:
        metabolite = model.metabolites[i]
        counter+=1
        
print(counter)

In [ ]:
# Save dead ends and their associated reaction in a pandas df
data = []

for i, is_dead_end in enumerate(dead_ends):
    if is_dead_end:
        metabolite = model.metabolites[i]
        reactions = [str(met_rxn) for met_rxn in metabolite.reactions]  # Convert reactions to strings
        data.append([metabolite.id] + reactions)

# Convert the list to a DataFrame
dead_ends_df = pd.DataFrame(data, columns=['Metabolite', 'Reaction1', 'Reaction2', 'Reaction3', 'Reaction4'])  # 'etc.' is a placeholder

# Adjusting the DataFrame to handle variable number of reactions
dead_ends_df = dead_ends_df.apply(lambda x: pd.Series(x.dropna().values), axis=1).fillna('')

# Renaming the columns appropriately
new_columns = ['Metabolite'] + [f'Reaction{i}' for i in range(1, len(dead_ends_df.columns))]
dead_ends_df.columns = new_columns

dead_ends_df.to_excel('temp/dead_ends_reactions.xlsx', index=False)

print(f'Total amount of dead-end metabolites: {len(dead_ends_df)}')  # To display the first few rows of the DataFrame

### 2.3 Gap-fill
Here we try different gap-filling approaches to fix dead-end metabolites in our reconstruction. First, we create artificial transport reactions if we detect that a dead-end metabolites is at oposite sides in different reactions. Then we extract boundary reactions from other reconstructions associated with our dead-end metabolites

In [ ]:
##### --- Create transport reactions to fill the gaps --- ######

def check_metabolite_sides(reactions, metabolite_base):
    sides = []  # List to store the side ('left' or 'right') of each reaction
    for reaction in reactions:
        # Splitting the reaction string into reactants and products
        if '-->' in reaction:
            reactants, products = reaction.split('-->')
        elif '<=>' in reaction:
            reactants, products = reaction.split('<=>')

        # Splitting reactants and products into individual metabolites and trimming whitespace
        reactant_ids = [r.strip() for r in reactants.split('+')]
        product_ids = [p.strip() for p in products.split('+')]

        # Constructing specific identifiers for comparison
        metabolite_id_with_compartment = f"{metabolite_base}_"

        # Checking if the specific identifier is in reactants or products
        if any(metabolite_id_with_compartment in r for r in reactant_ids):
            sides.append('left')
        if any(metabolite_id_with_compartment in p for p in product_ids):
            sides.append('right')

    return 'left' in sides and 'right' in sides


# Use the modified metabolites_compartments dictionary creation logic from before

# Initialize a dictionary to keep track of metabolites, their compartments, and reactions
metabolites_compartments = {}

for i, is_dead_end in enumerate(dead_ends):
    if is_dead_end:
        met = model.metabolites[i]
        base_id = met.id[:-2]  # Extract the base ID of the metabolite
        compartment = met.id[-1]  # Extract the compartment
        reactions = {str(met_rxn) for met_rxn in met.reactions}  # Use a set for unique reactions

        # Check if the base ID is already in the dictionary
        if base_id in metabolites_compartments:
            # Add the compartment if not already present and update the reactions set
            metabolites_compartments[base_id]['compartments'].add(compartment)
            metabolites_compartments[base_id]['reactions'].update(reactions)
        else:
            # If the base ID is not in the dictionary, add it with the current compartment and reactions
            metabolites_compartments[base_id] = {'compartments': {compartment}, 'reactions': reactions}

# Filtering metabolites present on opposite sides of reaction formulas
for metabolite, info in list(metabolites_compartments.items()):
    if len(info['compartments']) > 1:
        # Only keep metabolites that appear on opposite sides of the reaction equations
        if not check_metabolite_sides(info['reactions'], metabolite):
            del metabolites_compartments[metabolite]  # Remove metabolites not meeting the criteria

# Displaying the filtered results
counter=0
transport_reactions = []
for metabolite, info in metabolites_compartments.items():
    if len(info['compartments']) > 1:
        compartments = list(info['compartments'])
        for i in range(len(compartments)):
            for j in range(i+1, len(compartments)):
                # Construct the reaction string
                treaction = f"{metabolite}_{compartments[i]} <=> {metabolite}_{compartments[j]}"
                transport_reactions.append(treaction)
        print(f"{metabolite} is present in compartments: {', '.join(info['compartments'])}")
        print("Associated reactions:")
        for reaction in info['reactions']:
            print(reaction)
        print('------------------------------')
        print(f'Reaction created: {treaction}')
        counter+=1
        print()
print(counter)

In [ ]:
#Load models to extract boundary reactions from
iCHO1766 = read_sbml_model('../Data/Reconciliation/models/iCHOv1_final.xml')
iCHO2291 = read_sbml_model('../Data/Reconciliation/models/iCHO2291.xml')
recon3d = load_matlab_model('../Data/Reconciliation/models/Recon3D_301.mat')

models = [iCHO1766, iCHO2291, recon3d]

In [ ]:
###### --- Exctracting boundary reaction from other recosntructions --- ######

# Initialize the DataFrame with the desired columns
columns = ['ID','Name','Reaction', 'GPR', 'Subsystem', 'Lower Bound', 'Upper Bound']
reactions_df = pd.DataFrame(columns=columns)

# Initialize a set to track unique reaction IDs
seen_ids = set()

c=0
for mdl in models:
    for rxn in mdl.demands:
        # Standarize r.id according to our reconstruction
        rxn_id = rxn.id.replace('[', '_').replace(']', '').replace('_hs_', '_cho_').rstrip('_')
        if rxn_id not in seen_ids:
            seen_ids.add(rxn_id)
            # Standarize r.reaction according to our reconstruction
            rxn_reaction = rxn.reaction.replace('[', '_').replace(']', '').replace('_hs_', '_cho_')
            # Standarize met ids according to our reconstruction
            r_m = [m.id for m in rxn.metabolites][0].replace('[', '_').replace(']', '').replace('_hs_', '_cho_')
            for i, is_dead_end in enumerate(dead_ends):
                if is_dead_end:
                    met = model.metabolites[i]
                    if met.id == r_m:
                        # Create a temporary DataFrame for the new entry
                        new_row = pd.DataFrame({
                            'ID': [rxn_id],
                            'Name': [rxn.name],
                            'Reaction': [rxn_reaction],
                            'GPR': [rxn.gpr],
                            'Subsystem': [rxn.subsystem],
                            'Lower Bound': [rxn.lower_bound],
                            'Upper Bound': [rxn.upper_bound]
                        })
                        # Concatenate the new row to the main DataFrame
                        reactions_df = pd.concat([reactions_df, new_row], ignore_index=True)
                        c+=1

reactions_df.to_excel('temp/gap_fill_boundaries_output.xlsx', index=False)  # 'index=False' avoids writing row indices to the file.
print(f"Total reactions processed: {c}")

In [ ]:
###### --- Exctracting actual reactions from other reconstructions --- ######

# Initialize the DataFrame with the desired columns
columns = ['ID','Name','Reaction', 'GPR', 'Subsystem', 'Lower Bound', 'Upper Bound']
reactions_df = pd.DataFrame(columns=columns)

# Initialize a set to track unique reaction IDs
iCHO3000_rxn_ids = set([r.id for r in model.reactions])

c=0
for mdl in models:
    for rxn in mdl.reactions:
        # Standarize r.id according to our reconstruction
        rxn_id = rxn.id.replace('[', '_').replace(']', '').replace('_hs_', '_cho_').rstrip('_')
        if rxn_id not in iCHO3000_rxn_ids:
            iCHO3000_rxn_ids.add(rxn_id)
            # Standarize r.reaction according to our reconstruction
            rxn_reaction = rxn.reaction.replace('[', '_').replace(']', '').replace('_hs_', '_cho_')
            # Standarize met ids according to our reconstruction
            r_m = [m.id.replace('[', '_').replace(']', '').replace('_hs_', '_cho_') for m in rxn.metabolites]
            for i, is_dead_end in enumerate(dead_ends):
                if is_dead_end:
                    met = model.metabolites[i]
                    if met.id in r_m:
                        # Create a temporary DataFrame for the new entry
                        new_row = pd.DataFrame({
                            'ID': [rxn_id],
                            'Name': [rxn.name],
                            'Reaction': [rxn_reaction],
                            'GPR': [rxn.gpr],
                            'Subsystem': [rxn.subsystem],
                            'Lower Bound': [rxn.lower_bound],
                            'Upper Bound': [rxn.upper_bound],
                            'Dead_end': [met.id],
                            'Model': [mdl.id],
                        })
                        # Concatenate the new row to the main DataFrame
                        reactions_df = pd.concat([reactions_df, new_row], ignore_index=True)
                        c+=1

reactions_df.to_excel('temp/gap_fill_boundaries_output.xlsx', index=False)  # 'index=False' avoids writing row indices to the file.
print(f"Total reactions processed: {c}")

### 2.3 Addition of Extracellular Exchange Reanctions
The following cell adds **EXTRACELLULAR EXCHANGE** reactions to the dead-end metabolites in the extracellular compartment from the list generated above.

In [ ]:
##### ----- Automatically add EXTRACELLULAR EXCHANGE reactions to the "BoundaryRxns" Sheet ----- #####
added_exchange = False
for i,j in enumerate(dead_ends):
    if j:
        if str(model.metabolites[i]).endswith('_e'):
            new_row_data = {'Curated': '', 'Reaction': 'EX_'+str(model.metabolites[i]), 'Reaction Name': 'Exchange of '+model.metabolites[i].name, 'Reaction Formula': str(model.metabolites[i])+' <=>', 'Subsystem': 'EXTRACELLULAR EXCHANGE',
                                    'Reversible': 1, 'Lower bound': -1000, 'Upper bound': 1000, 'Objective': 0}
            new_row_df = pd.DataFrame(new_row_data, index=[len(boundary_rxns)])
            boundary_rxns = pd.concat([boundary_rxns, new_row_df])
            added_exchange = True

#Check for duplicated reactions added to the boundary_rxns dataset, IF NOT: update the google sheet file
if added_exchange:
    if not boundary_rxns['Reaction'].duplicated().any() and not boundary_rxns['Reaction Formula'].duplicated().any():
        sheet.update_google_sheet(sheet_boundary, boundary_rxns)
        print("BoundaryRxns Google Sheet updated.")
    else:
        print('Duplicated values found in the dataset')

### 2.4 Gapfill for blocked reactions
Cobrapy has a gap filling implementation that is very similar to that of Reed et al. where we use a mixed-integer linear program to figure out the smallest number of reactions that need to be added for a user-defined collection of reactions, i.e. a universal model.

In [ ]:
import cobra
from cobra.flux_analysis import gapfill

#recon_3d = read_sbml_model("../Data/GPR_curation/Recon3D.xml")
#iCHO2291 = read_sbml_model("../Data/Reconciliation/models/iCHO2291.xml")
#universal = recon_3d.merge(iCHO2291)

In [ ]:
for blocked_reaction in blocked:
    model.objective = blocked_reaction
    model.optimize().objective_value
    try:
        solution = gapfill(model, iCHO2291, demand_reactions=True)
        print(blocked_reaction)
        print(solution)
    except Exception as e:
        print(f'Gapfill failed for {blocked_reaction}: {str(e)}')
        continue

### Test CHO - Recon GEM

In [ ]:
universal

In [ ]:
# iCHO_recon3dfrom cobra.io import read_sbml_model
# read_sbml_model(".xml")

model_EX = [i for i, rxn in enumerate(model.reactions) if 'EX_' in rxn.id]
model_SK = [i for i, rxn in enumerate(model.reactions) if 'SK_' in rxn.id]
model_DM = [i for i, rxn in enumerate(model.reactions) if 'DM_' in rxn.id]
for i in model_EX:
    model.reactions[i].bounds = -1000, 1000

for i in model_SK:
    model.reactions[i].bounds = -1000, 1000

for i in model_DM:
    model.reactions[i].bounds = 0, 1000
    

In [ ]:
model.objective = "biomass_cho" # 
sol1 = model.optimize()
print(sol1.objective_value)

model.objective = "biomass_cho_prod" # 
sol2 = model.optimize()
print(sol2.objective_value)

In [ ]:
##### ----- Test model KOs ----- #####
for reaction in model.reactions:
    with model as model:
        reaction.knock_out()
        model.optimize()
        print('%s blocked (bounds: %s), new growth rate %f' %
              (reaction.id, str(reaction.bounds), model.objective.value))